# External-benchmark evaluation of the Santali systems on IN22-Gen (English → Santali)

**Purpose.** All earlier results are on a held-out split drawn from the MMLoSo training pool. This notebook evaluates the **already trained** Santali systems on **IN22-Gen**, the independent, multi-domain general benchmark released with IndicTrans2, to test whether the vocabulary-extension gain survives a change of domain. **No training happens here.**

**Systems evaluated (whatever checkpoints you attach):** pretrained model without adaptation; baseline (original vocabulary + LoRA) seeds 42 and 43; 445-token extension with mean initialisation (seeds 42 and 43) and random initialisation (seed 42); optionally the 1,220-token extension. All use the identical pipeline: `IndicProcessor` prompts, batch size 8, 4 beams, `max_length=256`, early stopping, one decoder, one scoring script.

**Outputs:** BLEU, chrF, chrF++ and TER per system; paired bootstrap of every extended run against both baseline seeds; subset analysis (reference contains an added word / output uses an added token); coverage of the added words in IN22-Gen; an overlap check between IN22-Gen and the training data.

**Not applicable to Bhili.** IN22-Gen covers the 22 scheduled languages, which do not include Bhili, so the Bhili systems cannot be evaluated on it.

## Inputs and settings required before running
| Item | Required | Notes |
|---|---|---|
| Notebook setting **Internet = On** | yes | pip install, model and IN22-Gen download |
| Notebook setting **Accelerator = GPU T4** | yes | one GPU is used |
| Kaggle secret **`HF_TOKEN`** | yes | your Hugging Face token |
| Dataset **Common Tribal Preprocessing** (`splits/santali_train.parquet`) | yes | used to rebuild the added-word lists and for the overlap check |
| Dataset with **baseline seed 42 adapter**: folder `baseline_v2_scaled/best_adapter/` (contains `adapter_model.safetensors`) | for that system | from your baseline v2 results zip |
| Dataset with **baseline seed 43**: `base_s43/best_trainable.pt` | for that system | from that run's results/backup zip |
| Dataset with the **extended runs**: `ext_f20_mean_s42/best_trainable.pt`, `ext_f20_mean_s43/best_trainable.pt`, `ext_f20_random_s42/best_trainable.pt` (optional `ext_f10_mean_s42/best_trainable.pt`) | for those systems | from each run's results/backup zip |

Missing checkpoints are skipped with a message, so the notebook runs with whatever subset you attach. **Run order:** run the install cell (Section 1.1), restart the session, then run everything from Section 1.2 onward. **Time:** about 1 hour (about 7 minutes of generation per system).

## Section 1. Environment
Same pinned packages as the training notebooks, with the incompatible preinstalled `torchao` removed. Keep the printed versions for your methods notes.

In [ ]:
# Section 1.1: install pinned dependencies (run ONCE, then restart the session)
%pip install -q --no-cache-dir \
    "transformers==4.46.3" \
    "tokenizers==0.20.3" \
    "IndicTransToolkit==1.1.1" \
    peft accelerate sacrebleu sentencepiece kagglehub datasets huggingface_hub

# Kaggle ships an old torchao that newer peft versions reject. This notebook does not use torchao.
%pip uninstall -y torchao

In [1]:
# Section 1.2: verify packages and GPU (run after the restart)
import sys
import importlib.metadata as metadata
import torch

print("=== Package versions ===")
for package in ["transformers", "tokenizers", "IndicTransToolkit", "torch", "peft", "accelerate", "sacrebleu", "sentencepiece", "datasets", "huggingface_hub", "torchao"]:
    try:
        print(f"{package:20s}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package:20s}: NOT INSTALLED")
print("\nPython:", sys.version.split()[0])
assert torch.cuda.is_available(), "No GPU. Set Accelerator to GPU T4 and restart the session."
print("GPU:", torch.cuda.get_device_name(0))
import transformers
assert transformers.__version__ == "4.46.3", "transformers is not 4.46.3: rerun the install cell and restart the session."

=== Package versions ===
transformers        : 4.46.3
tokenizers          : 0.20.3
IndicTransToolkit   : 1.1.1
torch               : 2.11.0+cu128
peft                : 0.20.0
accelerate          : 1.14.0
sacrebleu           : 2.6.0
sentencepiece       : 0.2.2
datasets            : 4.8.5
huggingface_hub     : 0.36.2
torchao             : NOT INSTALLED

Python: 3.13.15
GPU: Tesla T4


## Section 2. Authentication and configuration

In [2]:
# Section 2.1: imports, Hugging Face login, configuration
import gc, glob, json, os, re, shutil, time, types, unicodedata, warnings
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

warnings.filterwarnings("ignore")
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

DEVICE = torch.device("cuda")
MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"
SRC_LANG, TGT_LANG = "eng_Latn", "sat_Olck"
MAX_LEN = 256
BATCH_SIZE = 8
OUT_DIR = Path("/kaggle/working/in22gen_eval"); OUT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR = OUT_DIR / "predictions"; PRED_DIR.mkdir(exist_ok=True)
print("Output folder:", OUT_DIR)

Output folder: /kaggle/working/in22gen_eval


## Section 3. Data
**3.1** loads the Santali training split (used to rebuild the added-word lists and to check overlap). **3.2** downloads IN22-Gen from the Hugging Face hub. The dataset card documents pair configurations of the form `eng_Latn-hin_Deva` and an `all` configuration; this cell tries the Santali pair configuration first, falls back to the auto-converted parquet revision, then to `all`, prints the structure, and detects the English and Santali columns by name. **Read the printed structure and example before continuing**: the notebook stops with a clear message if the columns cannot be identified.

In [3]:
# Section 3.1: Santali training split (for candidate rebuild and overlap check)
INPUT_ROOT = Path("/kaggle/input")
matches = [p for p in INPUT_ROOT.rglob("santali_train.parquet") if p.parent.name == "splits"]
assert len(matches) == 1, f"Expected one splits/santali_train.parquet, found {len(matches)}. Attach the Common Tribal Preprocessing dataset."
train_df = pd.read_parquet(matches[0])
SRC_COL, TGT_COL = "English", "Santali"
assert (len(train_df), {SRC_COL, TGT_COL} <= set(train_df.columns)) == (17883, True)
train_texts_full = train_df[TGT_COL].dropna().astype(str).tolist()
print("Training sentences:", len(train_df))

Training sentences: 17883


In [6]:
# Section 3.2: download IN22-Gen (a GATED dataset: access must be granted to your Hugging Face account) and detect the English and Santali columns
import tarfile
from huggingface_hub import HfApi, hf_hub_download, whoami

REPO = "ai4bharat/IN22-Gen"
PARQUET = "data/train-00000-of-00001.parquet"
print("Hugging Face user for this token:", whoami(token=hf_token).get("name"))
print("Files in the repository:", HfApi().list_repo_files(REPO, repo_type="dataset", token=hf_token))

try:
    pq_path = hf_hub_download(REPO, PARQUET, repo_type="dataset", token=hf_token)
except Exception as e:
    msg = repr(e)
    if any(k in msg.lower() for k in ("gated", "403", "401", "restricted", "access")):
        raise RuntimeError(
            "IN22-Gen is gated and this token has no access yet. Fix: (1) log in to huggingface.co with the SAME account as your HF_TOKEN, "
            "(2) open https://huggingface.co/datasets/ai4bharat/IN22-Gen and accept the access conditions, (3) if you use a fine-grained token, "
            "enable access to gated public repositories (or create a classic 'Read' token), (4) update the HF_TOKEN secret, rerun Section 2.1, "
            "then rerun this cell.") from e
    raise

df_raw = pd.read_parquet(pq_path)
print("\nParquet shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))
print("First row:", {k: str(v)[:100] for k, v in df_raw.iloc[0].to_dict().items()})

def find_col(cols, lang):
    exact = [c for c in cols if c == lang or c == f"sentence_{lang}"]
    hit = exact or [c for c in cols if lang in c]
    return hit[0] if hit else None

cols = list(df_raw.columns)
src_col, tgt_col = find_col(cols, "eng_Latn"), find_col(cols, "sat_Olck")
if src_col is None or tgt_col is None:
    # fallback: list the raw benchmark archive so the structure can be inspected
    tar_path = hf_hub_download(REPO, "IN22_benchmark.tar.gz", repo_type="dataset", token=hf_token)
    extract_dir = Path("/kaggle/working/in22_raw"); extract_dir.mkdir(exist_ok=True)
    with tarfile.open(tar_path) as tf:
        tf.extractall(extract_dir)
    listing = sorted(str(p.relative_to(extract_dir)) for p in extract_dir.rglob("*") if p.is_file())
    print("\nThe parquet file does not contain both eng_Latn and sat_Olck columns. Files in the raw archive (first 80):")
    for f in listing[:80]:
        print("  ", f)
    raise RuntimeError("Could not find eng_Latn / sat_Olck columns in the parquet file. Send me the columns and the archive listing printed above.")

print("\nDetected source column:", src_col, "| reference column:", tgt_col)
in22_src_raw = df_raw[src_col].tolist()
in22_ref_raw = df_raw[tgt_col].tolist()

Hugging Face user for this token: sanjeevani06
Files in the repository: ['.gitattributes', 'IN22_benchmark.tar.gz', 'README.md', 'data/train-00000-of-00001.parquet']


data/train-00000-of-00001.parquet:   0%|          | 0.00/4.34M [00:00<?, ?B/s]


Parquet shape: (1024, 29)
Columns: ['context', 'source', 'url', 'domain', 'num_words', 'bucket', 'asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'eng_Latn', 'gom_Deva', 'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya', 'pan_Guru', 'san_Deva', 'sat_Olck', 'snd_Deva', 'tam_Taml', 'tel_Telu', 'urd_Arab']
First row: {'context': 'A uniform is often viewed as projecting a positive image of an organisation. Maintaining personal hy', 'source': 'web', 'url': 'None', 'domain': 'culture', 'num_words': '24', 'bucket': '18 - 25', 'asm_Beng': 'এটা বাহ্যিক ৰূপ হৈছে সশস্ত্র বাহিনীৰ জোৱানগৰাকীৰ সৈতে জড়িত কেইবাটাও বৈশিষ্ট, যেনে তেওঁৰ জোতা, কাপোৰ', 'ben_Beng': 'বাহ্য রূপ হল সেই পরিষেবা প্রদানকারী মানুষ সম্বন্ধীয় এক গুচ্ছ বৈশিষ্ট্য যেমন, তার জুতো, জামাকাপড়, টাই', 'brx_Deva': 'नुजानाया खामानिनि सुबुंजों सोमोन्दो गोनां महरफोरनि मोनसे जथाइ, जेरै बिसोरनि जुता, गस्ला, टाइ, गहेना,', 'doi_Deva': 'इक दखौट सेवा करने आह्\u200c\u200c\u200cले व्य

In [7]:
# Section 3.3: clean the benchmark and check overlap with the training data
def norm(s): return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s))).strip()

pairs = [(norm(s), norm(r)) for s, r in zip(in22_src_raw, in22_ref_raw)]
keep = [i for i, (s, r) in enumerate(pairs) if s and r]
in22_src = [pairs[i][0] for i in keep]
in22_ref = [pairs[i][1] for i in keep]
print(f"IN22-Gen sentences: {len(pairs)} | kept after removing empty pairs: {len(in22_src)}")
print("Example:\n  EN :", in22_src[0][:140], "\n  SAT:", in22_ref[0][:140])
assert len(in22_src) > 500, "Unexpectedly small benchmark: check the detected columns."
assert any("\u1C50" <= ch <= "\u1C7F" for ch in "".join(in22_ref[:50])), "References do not contain Ol Chiki characters: wrong column?"

train_src_set = {norm(s) for s in train_df[SRC_COL].dropna().astype(str)}
train_ref_set = {norm(s) for s in train_df[TGT_COL].dropna().astype(str)}
ov_src = np.array([s in train_src_set for s in in22_src])
ov_ref = np.array([r in train_ref_set for r in in22_ref])
print(f"Overlap with the training split (exact match after normalisation): English sources {int(ov_src.sum())}, Santali references {int(ov_ref.sum())}, either {int((ov_src | ov_ref).sum())}")
clean_mask = ~(ov_src | ov_ref)
print("Sentences used for the 'clean' (no-overlap) scores:", int(clean_mask.sum()))
pd.DataFrame({"source": in22_src, "reference": in22_ref}).to_csv(OUT_DIR / "in22gen_en_sat.csv", index=False, encoding="utf-8-sig")

IN22-Gen sentences: 1024 | kept after removing empty pairs: 1024
Example:
  EN : An appearance is a bunch of attributes related to the service person, like their shoes, clothes, tie, jewellery, hairstyle, make-up, watch,  
  SAT: ᱢᱩᱴᱷᱟᱹᱱ ᱫᱚ ᱦᱩᱭᱩᱜ ᱠᱟᱱᱟ ᱪᱟᱹᱠᱨᱤᱭᱟᱱ ᱦᱚᱲ ᱥᱟᱶᱛᱮ ᱥᱟᱹᱜᱟᱹᱭᱟᱱᱟᱜ ᱜᱩᱱ ᱠᱚ ᱨᱮᱭᱟᱜ ᱢᱤᱫ ᱛᱷᱚᱯᱮ ᱡᱮᱞᱮᱠᱟ ᱯᱚᱴᱚᱢ ᱯᱟᱱᱟᱦᱤ, ᱠᱤᱪᱨᱤᱪ ᱠᱷᱟᱹᱸᱰᱣᱟᱹᱜ, ᱴᱟᱭ, ᱜᱚᱭᱱᱟ, ᱩᱯ ᱨᱮᱭᱟᱜ ᱥᱴ
Overlap with the training split (exact match after normalisation): English sources 1, Santali references 0, either 1
Sentences used for the 'clean' (no-overlap) scores: 1023


## Section 4. Tokenizer and source preprocessor
The same IndicTrans2 tokenizer and `IndicProcessor` as in all training runs. The tokenizer is unchanged; extended systems differ only in how generated IDs above 122,671 are mapped back to words.

In [8]:
# Section 4.1: tokenizer and IndicProcessor
from transformers import AutoTokenizer
from IndicTransToolkit import IndicProcessor

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, token=hf_token)
processor = IndicProcessor(inference=True)
sp = tok.tgt_spm
BASE = len(tok.tgt_encoder)
EOS_ID, PAD_ID, DEC_START_ID, UNK_ID = 2, 1, 2, 3
assert BASE == 122672 and (tok.eos_token_id, tok.pad_token_id) == (EOS_ID, PAD_ID)
print("Tokenizer:", type(tok).__name__, "| target vocabulary:", BASE)

# prompts are computed once and shared by all systems
prompts_all = [processor.preprocess_batch([s], src_lang=SRC_LANG, tgt_lang=TGT_LANG)[0] for s in in22_src]
print("Prompts prepared:", len(prompts_all), "| example:", prompts_all[0][:100])

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-indic-1B:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

dict.TGT.json: 0.00B [00:00, ?B/s]

model.SRC:   0%|          | 0.00/3.26M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

Tokenizer: IndicTransTokenizer | target vocabulary: 122672
Prompts prepared: 1024 | example: eng_Latn sat_Olck An appearance is a bunch of attributes related to the service person , like their 


## Section 5. Added-word lists and extension encoder/decoder
The added-word lists are rebuilt from the training split with exactly the rule used in the training runs (whole whitespace words, frequency at least `MIN_FREQ`, no punctuation or symbols, more than one SentencePiece piece, ordered by frequency, then piece saving, then spelling). The counts must match the runs (445 for `MIN_FREQ=20`, 1,220 for `MIN_FREQ=10`). The decoder is the continuation-aware decoder used in the final evaluation; the checks below must pass.

In [9]:
# Section 5.1: rebuild candidate lists from the TRAINING split
word_counts = Counter()
for t in train_texts_full:
    word_counts.update(t.split())

def _only_punct(w): return all(unicodedata.category(c)[0] in {"P", "S"} for c in w)
def _word_only(w): return bool(w) and all(unicodedata.category(c)[0] not in {"P", "S"} for c in w)

rows = []
for w, f in word_counts.items():
    if f < 5 or _only_punct(w):
        continue
    pieces = sp.encode(w, out_type=str)
    if len(pieces) > 1:
        rows.append({"word": w, "frequency": f, "piece_count": len(pieces), "potential_piece_reduction": len(pieces) - 1})
clean_candidates = pd.DataFrame(rows)
clean_candidates = clean_candidates[clean_candidates["word"].map(_word_only)].copy()
_existing = set(tok.tgt_encoder.keys())

def select_candidates(min_freq):
    sub = (clean_candidates[(clean_candidates["frequency"] >= min_freq) & (clean_candidates["piece_count"] > 1)]
           .drop_duplicates(subset=["word"])
           .sort_values(["frequency", "potential_piece_reduction", "word"], ascending=[False, False, True], kind="stable")
           .reset_index(drop=True))
    return [w for w in sub["word"].tolist() if w not in _existing]

WORDS = {445: select_candidates(20), 1220: select_candidates(10)}
for n, w in WORDS.items():
    print(f"{n} tokens expected -> rebuilt {len(w)}")
    assert len(w) == n, "Candidate count differs from the training runs: check the dataset version."
print("PASS: candidate lists match the training runs.")

445 tokens expected -> rebuilt 445
1220 tokens expected -> rebuilt 1220
PASS: candidate lists match the training runs.


In [11]:
# Diagnostic: which sentences differ for the 1,220-token decoder, and is the difference only whitespace?
def nows(s): return re.sub(r"\s+", "", s)
e = EXT[1220]
missing_words = {w for w in e.words if has_missing_piece(w)}
ws_only, real, shown = 0, 0, 0
for t in check_texts:
    if any(m.group(1) in missing_words for m in e.pattern.finditer(t)):
        continue
    a = tok.decode(encode_original_target_segment(t), skip_special_tokens=True, clean_up_tokenization_spaces=True).strip()
    b = e.decode([DEC_START_ID] + e.encode(t) + [EOS_ID])
    if a != b:
        kind = "whitespace-only" if nows(a) == nows(b) else "TEXT DIFFERS"
        ws_only += (kind == "whitespace-only"); real += (kind != "whitespace-only")
        if shown < 6:
            shown += 1
            k = next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), min(len(a), len(b)))
            print(f"[{kind}]\n  TEXT: {t[:150]}\n  ORIG: {a[:150]}\n  EXT : {b[:150]}")
            print("  first difference at char", k, "| orig:", repr(a[max(0, k-15):k+15]), "| ext:", repr(b[max(0, k-15):k+15]), "\n")
print("whitespace-only differences:", ws_only, "| text differences:", real)

[whitespace-only]
  TEXT: ᱠᱩᱢᱟᱨ ᱞᱮᱠᱟᱱ ᱢᱤᱫᱴᱟᱝ ᱥᱮᱨᱮᱧᱤᱭᱟᱹᱼᱟᱜ ᱛᱚᱞᱟᱥ ᱨᱮ ᱥᱚᱱᱤᱼᱴᱤᱵᱷᱤ ᱥᱟᱹᱜᱤᱧ ᱧᱮᱱᱮᱞᱚᱢ ᱢᱤᱫ ᱦᱮᱯᱨᱟᱣ 'ᱠᱮ ᱯᱷᱚᱨ ᱠᱤᱥᱳᱨ' ᱮ ᱥᱟᱯᱲᱟᱣ ᱞᱮᱫᱟ ᱾
  ORIG: ᱠᱩᱢᱟᱨ ᱞᱮᱠᱟᱱ ᱢᱤᱫᱴᱟᱝ ᱥᱮᱨᱮᱧᱤᱭᱟᱹᱼᱟᱜ ᱛᱚᱞᱟᱥ ᱨᱮ ᱥᱚᱱᱤᱼᱴᱤᱵᱷᱤ ᱥᱟᱹᱜᱤᱧ ᱧᱮᱱᱮᱞᱚᱢ ᱢᱤᱫ ᱦᱮᱯᱨᱟᱣ 'ᱠᱮ ᱯᱷᱚᱨ ᱠᱤᱥᱳᱨ' ᱮ ᱥᱟᱯᱲᱟᱣ ᱞᱮᱫᱟ ᱾
  EXT : ᱠᱩᱢᱟᱨ ᱞᱮᱠᱟᱱ ᱢᱤᱫᱴᱟᱝ ᱥᱮᱨᱮᱧᱤᱭᱟᱹᱼᱟᱜ ᱛᱚᱞᱟᱥ ᱨᱮ ᱥᱚᱱᱤᱼᱴᱤᱵᱷᱤ ᱥᱟᱹᱜᱤᱧ ᱧᱮᱱᱮᱞᱚᱢ ᱢᱤᱫ ᱦᱮᱯᱨᱟᱣ 'ᱠᱮ ᱯᱷᱚᱨ ᱠᱤᱥᱳᱨ ' ᱮ ᱥᱟᱯᱲᱟᱣ ᱞᱮᱫᱟ ᱾
  first difference at char 92 | orig: " 'ᱠᱮ ᱯᱷᱚᱨ ᱠᱤᱥᱳᱨ' ᱮ ᱥᱟᱯᱲᱟᱣ ᱞᱮᱫᱟ" | ext: " 'ᱠᱮ ᱯᱷᱚᱨ ᱠᱤᱥᱳᱨ ' ᱮ ᱥᱟᱯᱲᱟᱣ ᱞᱮᱫ" 

[whitespace-only]
  TEXT: ᱢᱤᱫᱫᱷᱟᱣᱮ ᱞᱟᱹᱭᱞᱮᱫᱟ, 'ᱤᱧ ᱫᱚᱧ ᱠᱚᱨᱟᱣᱟ', ᱱᱤᱭᱟᱹ ᱠᱚᱨᱟᱣ ᱵᱮᱜᱚᱨ ᱟᱢᱟᱜᱽ ᱟᱨ ᱮᱴᱟᱜ ᱩᱯᱟᱹᱭ ᱵᱟᱹᱱᱩᱜᱼᱟ ᱾
  ORIG: ᱢᱤᱫᱫᱷᱟᱣᱮ ᱞᱟᱹᱭᱞᱮᱫᱟ, 'ᱤᱧ ᱫᱚᱧ ᱠᱚᱨᱟᱣᱟ', ᱱᱤᱭᱟᱹ ᱠᱚᱨᱟᱣ ᱵᱮᱜᱚᱨ ᱟᱢᱟᱜᱽ ᱟᱨ ᱮᱴᱟᱜ ᱩᱯᱟᱹᱭ ᱵᱟᱹᱱᱩᱜᱼᱟ ᱾
  EXT : ᱢᱤᱫᱫᱷᱟᱣᱮ ᱞᱟᱹᱭᱞᱮᱫᱟ, 'ᱤᱧ ᱫᱚᱧ ᱠᱚᱨᱟᱣᱟ ', ᱱᱤᱭᱟᱹ ᱠᱚᱨᱟᱣ ᱵᱮᱜᱚᱨ ᱟᱢᱟᱜᱽ ᱟᱨ ᱮᱴᱟᱜ ᱩᱯᱟᱹᱭ ᱵᱟᱹᱱᱩᱜᱼᱟ ᱾
  first difference at char 33 | orig: " 'ᱤᱧ ᱫᱚᱧ ᱠᱚᱨᱟᱣᱟ', ᱱᱤᱭᱟᱹ ᱠᱚᱨᱟᱣ " | ext: " 'ᱤᱧ ᱫᱚᱧ ᱠᱚᱨᱟᱣᱟ ', ᱱᱤᱭᱟᱹ ᱠᱚᱨᱟᱣ" 

[whitespace-only]
  TEXT: ᱞᱮᱛᱟᱲ ᱨᱤᱠᱟᱹ ᱦᱚᱛᱮᱛᱮ ᱜᱞᱳᱵᱟᱞ ᱤ ᱳ ᱰᱤ ᱵᱤ ᱨᱮᱝᱠᱤᱝᱨᱮ ᱟᱵᱚᱣᱟᱜ ᱨᱮᱴᱤᱝ ᱨᱟᱠᱟᱵ ᱟᱠᱟᱱᱟ ᱟᱨ ᱡᱤ ᱵ

In [12]:
# integrity checks on training text and IN22-Gen references
check_texts = [t.strip() for t in train_texts_full[:300]] + in22_ref[:300]
for n in (445, 1220):
    e = EXT[n]
    missing_words = {w for w in e.words if has_missing_piece(w)}
    unchanged = mism = compared = diff_ws = diff_text = skipped = 0
    for t in check_texts:
        if e.pattern.search(t) is None:
            unchanged += 1
            mism += (e.encode(t) != encode_original_target_segment(t))
        if any(m.group(1) in missing_words for m in e.pattern.finditer(t)):
            skipped += 1
            continue
        compared += 1
        a = tok.decode(encode_original_target_segment(t), skip_special_tokens=True, clean_up_tokenization_spaces=True).strip()
        b = e.decode([DEC_START_ID] + e.encode(t) + [EOS_ID])
        if a != b:
            if re.sub(r"\s+", "", a) == re.sub(r"\s+", "", b):
                diff_ws += 1
            else:
                diff_text += 1
    print(f"{n} tokens: unchanged-text mismatches {mism}/{unchanged} (must be 0); decoder differences: whitespace-only {diff_ws}, "
          f"text {diff_text} of {compared} (text must be 0); {skipped} sentences skipped (added word with a missing piece; {len(missing_words)} such words)")
    assert mism == 0 and diff_text == 0
    assert diff_ws <= max(2, 0.02 * compared), "Too many whitespace-only differences: inspect them."
print("PASS: encoder and decoder are consistent (whitespace-only differences are reported above).")

445 tokens: unchanged-text mismatches 0/261 (must be 0); decoder differences: whitespace-only 0, text 0 of 600 (text must be 0); 0 sentences skipped (added word with a missing piece; 0 such words)
1220 tokens: unchanged-text mismatches 0/143 (must be 0); decoder differences: whitespace-only 4, text 0 of 598 (text must be 0); 2 sentences skipped (added word with a missing piece; 3 such words)
PASS: encoder and decoder are consistent (whitespace-only differences are reported above).


## Section 6. How much of IN22-Gen do the added words cover?
The training-domain extension can only help where the added words occur. This CPU-only check measures their coverage in the IN22-Gen references and compares it with the MMLoSo test split (about 5.6% of reference words for the 445-token list). A much lower coverage predicts a proportionally smaller effect before any model is run.

In [13]:
# Section 6.1: coverage of the added words, token reduction and unknown-token rate on the IN22-Gen references
mmloso_test = pd.read_parquet(matches[0].parent / "santali_test.parquet")[TGT_COL].dropna().astype(str).str.strip().tolist()

def coverage(refs, e):
    total_words = sum(len(r.split()) for r in refs)
    hits = sum(len(e.pattern.findall(r)) for r in refs)
    return hits, total_words

rows = []
for n in (445, 1220):
    e = EXT[n]
    for label, refs in [("IN22-Gen references", in22_ref), ("MMLoSo test references", mmloso_test)]:
        hits, tw = coverage(refs, e)
        o = sum(len(encode_original_target_segment(r)) for r in refs)
        x = sum(len(e.encode(r)) for r in refs)
        rows.append({"added list": f"{n} words", "text": label, "reference words": tw, "added-word occurrences": hits,
                     "coverage %": round(100 * hits / tw, 2), "token reduction %": round(100 * (o - x) / o, 2)})
display(pd.DataFrame(rows))

unk = sum(sum(1 for t in encode_original_target_segment(r) if t == UNK_ID) for r in in22_ref)
tot = sum(len(encode_original_target_segment(r)) for r in in22_ref)
print(f"Unknown-token rate of the IN22-Gen references under the original tokenizer: {100*unk/tot:.2f}%")
pd.DataFrame(rows).to_csv(OUT_DIR / "coverage.csv", index=False)

,added list,text,reference words,added-word occurrences,coverage %,token reduction %
0,445 words,IN22-Gen references,26676,880,3.30,2.50
1,445 words,MMLoSo test references,19323,1085,5.62,4.92
2,1220 words,IN22-Gen references,26676,1689,6.33,4.77
3,1220 words,MMLoSo test references,19323,1583,8.19,7.37


Unknown-token rate of the IN22-Gen references under the original tokenizer: 2.44%


## Section 7. Model builder and weight loaders
One fp16 base model is built per vocabulary size and the trained weights of each system are loaded into it in turn. Extended models replace the decoder embedding and output layer with the split-weight modules used in training (original rows frozen, new rows in fp32) and keep the fp16-safe forward. The pretrained, unadapted system is evaluated first: a freshly attached LoRA adapter has zero output weights, so it equals the pretrained model (checked by summing the absolute adapter weights).

In [17]:
# Section 7.1: split-weight modules, model builder, loaders
from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict, TaskType
from safetensors.torch import load_file

class ExtEmbedding(nn.Module):
    def __init__(self, original_weight, new_weight, padding_idx=None):
        super().__init__()
        self.padding_idx = padding_idx
        self.embedding_dim = original_weight.shape[1]
        self.original_weight = nn.Parameter(original_weight.detach().clone(), requires_grad=False)
        self.new_weight = nn.Parameter(new_weight.detach().clone().float(), requires_grad=True)
    def forward(self, input_ids):
        w = torch.cat([self.original_weight, self.new_weight.to(self.original_weight.dtype)], dim=0)
        return F.embedding(input_ids, w, padding_idx=self.padding_idx)

class ExtLMHead(nn.Module):
    def __init__(self, original_weight, new_weight, bias=None):
        super().__init__()
        self.original_weight = nn.Parameter(original_weight.detach().clone(), requires_grad=False)
        self.new_weight = nn.Parameter(new_weight.detach().clone().float(), requires_grad=True)
        if bias is None:
            self.register_parameter("bias", None)
        else:
            self.bias = nn.Parameter(bias.detach().clone(), requires_grad=False)
    def forward(self, hidden_states):
        w = torch.cat([self.original_weight, self.new_weight.to(self.original_weight.dtype)], dim=0).to(hidden_states.dtype)
        b = self.bias.to(hidden_states.dtype) if self.bias is not None else None
        return F.linear(hidden_states, w, b)

def build_model(n_new):
    base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, trust_remote_code=True, token=hf_token,
                                                 torch_dtype=torch.float16, low_cpu_mem_usage=True).to(DEVICE)
    base.config.vocab_size = BASE + n_new
    base.config.decoder_vocab_size = BASE + n_new
    cfg = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none",
                     task_type=TaskType.SEQ_2_SEQ_LM)
    lm = get_peft_model(base, cfg)
    if n_new > 0:
        inner = lm.base_model.model
        dec = inner.model.decoder
        old_emb, old_head = dec.embed_tokens, inner.lm_head
        assert old_emb.weight.shape[0] == BASE and old_head.weight.shape[0] == BASE
        zeros = torch.zeros(n_new, old_emb.weight.shape[1], dtype=torch.float32, device=old_emb.weight.device)
        dec.embed_tokens = ExtEmbedding(old_emb.weight.detach(), zeros, padding_idx=old_emb.padding_idx).to(DEVICE)
        inner.lm_head = ExtLMHead(old_head.weight.detach(), zeros.clone(), bias=old_head.bias).to(DEVICE)
        del old_emb, old_head
    for p in lm.parameters():
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()
    gc.collect(); torch.cuda.empty_cache()
    return lm.eval()

def lora_b_sum(lm):
    return sum(p.detach().abs().sum().item() for n, p in lm.named_parameters() if "lora_B" in n)

def weight_signature(lm):
    return round(sum(p.detach().float().abs().sum().item() for p in lm.parameters() if p.requires_grad), 3)


def load_trainable_state(lm, path, n_new):
    sd = torch.load(path, map_location="cpu", weights_only=False)

    params = dict(lm.named_parameters())

    # Match checkpoint keys exactly first. If wrapper prefixes differ,
    # allow a unique suffix match with an identical tensor shape.
    mapped = {}
    missing = []
    used_params = set()

    for ckpt_key, value in sd.items():
        if not torch.is_tensor(value):
            continue

        if ckpt_key in params:
            model_key = ckpt_key
        else:
            # The baseline seed-43 checkpoint contains empty extension
            # tensors. The original-vocabulary model does not need them.
            if n_new == 0 and ckpt_key.endswith("new_weight"):
                assert value.numel() == 0, (
                    f"Unexpected non-empty extension tensor: {ckpt_key}"
                )
                continue

            candidates = [
                k for k, p in params.items()
                if (k.endswith(ckpt_key) or ckpt_key.endswith(k))
                and tuple(p.shape) == tuple(value.shape)
            ]

            if len(candidates) != 1:
                missing.append(
                    (ckpt_key, tuple(value.shape), candidates[:5])
                )
                continue

            model_key = candidates[0]

        assert model_key not in used_params, (
            f"Multiple checkpoint tensors map to {model_key}"
        )
        assert tuple(params[model_key].shape) == tuple(value.shape), (
            f"Shape mismatch for {model_key}: "
            f"{tuple(value.shape)} vs {tuple(params[model_key].shape)}"
        )

        mapped[model_key] = value
        used_params.add(model_key)

    assert not missing, (
        "Could not uniquely map checkpoint keys. "
        f"First examples: {missing[:5]}"
    )

    # Every LoRA parameter must be loaded.
    lora_keys = [k for k in params if "lora_" in k]
    missing_lora = [k for k in lora_keys if k not in mapped]
    assert not missing_lora, (
        f"LoRA parameters missing from checkpoint: {missing_lora[:5]}"
    )

    # Ensure the extended embeddings are present for extended models.
    if n_new > 0:
        ext_keys = [
            k for k in params
            if k.endswith("new_weight")
        ]
        assert len(ext_keys) == 2, (
            f"Expected two extension tensors; found {ext_keys}"
        )

        for k in ext_keys:
            assert k in mapped, f"Extension weights were not loaded: {k}"
            assert mapped[k].shape[0] == n_new, (
                f"{k}: expected {n_new} added rows, "
                f"got {mapped[k].shape[0]}"
            )

    # Copy only after every validation has passed.
    with torch.no_grad():
        for k, value in mapped.items():
            params[k].copy_(
                value.to(device=params[k].device, dtype=params[k].dtype)
            )

    print(
        f"Loaded {len(mapped)} tensors from {Path(path).name}; "
        f"LoRA tensors: {len(lora_keys)}; added tokens: {n_new}"
    )

def load_peft_adapter(lm, adapter_dir):
    sd = load_file(str(Path(adapter_dir) / "adapter_model.safetensors"))
    res = set_peft_model_state_dict(lm, sd)
    missing = [k for k in (getattr(res, "missing_keys", []) or []) if "lora_" in k]
    assert not missing, f"LoRA weights missing after loading the adapter: {missing[:3]}"
print("Builders and loaders defined.")

Builders and loaders defined.


## Section 8. Locate the trained checkpoints
Each system is searched for under `/kaggle/input` by folder name. Systems whose checkpoint is not found are skipped (listed below), so attach whichever checkpoints you have.

In [15]:
# Section 8.1: systems and their checkpoints (supports flat file names such as f20_mean_s42best_trainable.pt and run-named folders)
def find_ckpt(candidates, kind):
    for cand in candidates:
        if kind == "peft":     # a folder that contains adapter_model.safetensors (searched at any depth below it)
            hits = glob.glob(f"/kaggle/input/**/{cand}/**/adapter_model.safetensors", recursive=True)
            if hits:
                return str(Path(hits[0]).parent)
        else:                  # a .pt file, either by exact file name or as best_trainable.pt inside a run folder
            hits = (glob.glob(f"/kaggle/input/**/{cand}", recursive=True) if cand.endswith(".pt")
                    else glob.glob(f"/kaggle/input/**/{cand}/best_trainable.pt", recursive=True))
            if hits:
                return hits[0]
    return None

SYSTEMS = [  # (display name, kind, added tokens, name candidates)
    ("pretrained, no adaptation", "zero", 0, []),
    ("baseline seed 42", "peft", 0, ["baseline_v2_scaled_best_adapter", "baseline_v2_scaled/best_adapter", "best_adapter"]),
    ("baseline seed 43", "pt", 0, ["base_s43_best_trainable.pt", "base_s43best_trainable.pt", "base_s43", "baseline_s43"]),
    ("mean init 445, seed 42", "pt", 445, ["f20_mean_s42best_trainable.pt", "f20_mean_s42_best_trainable.pt", "ext_f20_mean_s42"]),
    ("mean init 445, seed 43", "pt", 445, ["f20_mean_s43best_trainable.pt", "f20_mean_s43_best_trainable.pt", "ext_f20_mean_s43"]),
    ("random init 445, seed 42", "pt", 445, ["f20_random_s42best_trainable.pt", "f20_random_s42_best_trainable.pt", "ext_f20_random_s42"]),
    ("mean init 1220, seed 42", "pt", 1220, ["f10_mean_s42best_trainable.pt", "f10_mean_s42_best_trainable.pt", "ext_f10_mean_s42"]),
]
plan, rows = [], []
for name, kind, n_new, cands in SYSTEMS:
    path = None if kind == "zero" else find_ckpt(cands, kind)
    ok = kind == "zero" or path is not None
    rows.append({"system": name, "added tokens": n_new, "checkpoint": "(none: pretrained)" if kind == "zero" else (path or "NOT FOUND (skipped)")})
    if ok:
        plan.append((name, kind, n_new, path))
display(pd.DataFrame(rows))
assert len(plan) >= 2, "Too few checkpoints found. Attach the baseline and extended run files as a dataset."

,system,added tokens,checkpoint
0,"pretrained, no adaptation",0,(none: pretrained)
1,baseline seed 42,0,/kaggle/input/datasets/sanjeevanijaiswal/train...
2,baseline seed 43,0,/kaggle/input/datasets/sanjeevanijaiswal/train...
3,"mean init 445, seed 42",445,/kaggle/input/datasets/sanjeevanijaiswal/train...
4,"mean init 445, seed 43",445,/kaggle/input/datasets/sanjeevanijaiswal/train...
5,"random init 445, seed 42",445,/kaggle/input/datasets/sanjeevanijaiswal/train...
6,"mean init 1220, seed 42",1220,NOT FOUND (skipped)


## Section 9. Generation on IN22-Gen
Every system translates the same IN22-Gen English sentences through the identical pipeline. Predictions are written to CSV as they finish, so an interrupted session keeps completed systems. A weight signature is printed per system to confirm that the weights really changed between systems.

In [18]:

# Section 9.1: resumable generation
# Reuse saved predictions and generate only missing systems.

@torch.inference_mode()
def generate_all(lm, ext, batch_size=BATCH_SIZE):
    texts, id_lists = [], []

    for i in range(0, len(prompts_all), batch_size):
        enc = tok(
            prompts_all[i:i + batch_size],
            padding=True,
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        ).to(DEVICE)

        gen = lm.generate(
            **enc,
            max_length=256,
            num_beams=4,
            early_stopping=True,
            do_sample=False
        )

        for row in gen.cpu().tolist():
            id_lists.append(row)
            texts.append(ext.decode(row))

        if (i // batch_size) % 25 == 0:
            print(
                f"   {len(texts)}/{len(prompts_all)}",
                end="\r"
            )

    return texts, id_lists


def safe_system_name(name):
    return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")


# Restore completed predictions from disk.
results = {}

for name, kind, n_new, path in plan:
    safe = safe_system_name(name)
    csv_path = PRED_DIR / f"{safe}.csv"
    ids_path = PRED_DIR / f"{safe}_generated_ids.json"

    if csv_path.exists() and ids_path.exists():
        df = pd.read_csv(csv_path)
        ids = json.load(open(ids_path, encoding="utf-8"))

        # Check that the saved predictions match this benchmark.
        assert len(df) == len(in22_src), (
            f"Saved prediction count mismatch for {name}"
        )
        assert len(ids) == len(in22_src), (
            f"Saved generated-ID count mismatch for {name}"
        )
        assert df["source"].astype(str).tolist() == in22_src, (
            f"Saved source order differs for {name}"
        )
        assert df["reference"].astype(str).tolist() == in22_ref, (
            f"Saved reference order differs for {name}"
        )

        hyp = df["prediction"].fillna("").astype(str).tolist()
        results[name] = {
            "hyp": hyp,
            "ids": ids,
            "n_new": n_new
        }

        print(f"[CACHED] {name}: reusing saved predictions")


# Generate only systems without valid saved predictions.
for name, kind, n_new, path in plan:
    if name in results:
        continue

    print(f"\n=== Evaluating {name} ===")

    # Fresh model per system prevents accidental weight carry-over.
    lm = build_model(n_new)

    try:
        if kind == "zero":
            assert lora_b_sum(lm) == 0, (
                "Fresh LoRA adapter is not an identity adapter."
            )

        elif kind == "peft":
            load_peft_adapter(lm, path)

        else:
            load_trainable_state(lm, path, n_new)

        print(
            f"Weight signature: {weight_signature(lm)} | "
            f"sum|lora_B|: {lora_b_sum(lm):.2f}"
        )

        t0 = time.time()
        hyp, ids = generate_all(lm, EXT[n_new])

        assert len(hyp) == len(in22_src)
        assert len(ids) == len(in22_src)

        results[name] = {
            "hyp": hyp,
            "ids": ids,
            "n_new": n_new
        }

        safe = safe_system_name(name)

        pd.DataFrame({
            "source": in22_src,
            "reference": in22_ref,
            "prediction": hyp
        }).to_csv(
            PRED_DIR / f"{safe}.csv",
            index=False,
            encoding="utf-8-sig"
        )

        with open(
            PRED_DIR / f"{safe}_generated_ids.json",
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(ids, f)

        print(
            f"\nCompleted {name} in "
            f"{(time.time() - t0) / 60:.1f} minutes"
        )

    finally:
        del lm
        gc.collect()
        torch.cuda.empty_cache()

print("\nSystems with predictions:", list(results))


[CACHED] pretrained, no adaptation: reusing saved predictions
[CACHED] baseline seed 42: reusing saved predictions

=== Evaluating baseline seed 43 ===
Loaded 216 tensors from base_s43_best_trainable.pt; LoRA tensors: 216; added tokens: 0
Weight signature: 53925.093 | sum|lora_B|: 14323.18
   1008/1024
Completed baseline seed 43 in 8.2 minutes

=== Evaluating mean init 445, seed 42 ===
Loaded 218 tensors from f20_mean_s42best_trainable.pt; LoRA tensors: 216; added tokens: 445
Weight signature: 79163.456 | sum|lora_B|: 14181.98
   1008/1024
Completed mean init 445, seed 42 in 8.6 minutes

=== Evaluating mean init 445, seed 43 ===
Loaded 218 tensors from f20_mean_s43best_trainable.pt; LoRA tensors: 216; added tokens: 445
Weight signature: 79398.53 | sum|lora_B|: 14323.75
   1008/1024
Completed mean init 445, seed 43 in 8.3 minutes

=== Evaluating random init 445, seed 42 ===
Loaded 218 tensors from f20_random_s42best_trainable.pt; LoRA tensors: 216; added tokens: 445
Weight signature: 86

## Section 10. Scores
BLEU uses `sacrebleu` defaults. chrF is character-level; **chrF++** adds word bigrams (`word_order=2`) and is the metric the IndicTrans2 papers report. Note that the official IndicTrans2 evaluation tokenises Indic outputs with IndicNLP before scoring, so absolute BLEU here is not exactly the published protocol; all systems share the same scoring, so comparisons between them are valid. The 'clean' rows exclude any sentence that overlaps the training data (identical to the full rows if there is no overlap).

In [19]:
# Section 10.1: score table
import sacrebleu

def scores(hyp, ref):
    return {"BLEU": round(sacrebleu.corpus_bleu(hyp, [ref]).score, 2),
            "chrF": round(sacrebleu.corpus_chrf(hyp, [ref]).score, 2),
            "chrF++": round(sacrebleu.corpus_chrf(hyp, [ref], word_order=2).score, 2),
            "TER": round(sacrebleu.corpus_ter(hyp, [ref]).score, 2),
            "word_len_ratio": round(sum(len(h.split()) for h in hyp) / sum(len(r.split()) for r in ref), 3),
            "empty": sum(h == "" for h in hyp)}

rows = []
for name, r in results.items():
    rows.append({"system": name, "subset": "all", "n": len(in22_ref), **scores(r["hyp"], in22_ref)})
    if (~clean_mask).any():
        idx = np.where(clean_mask)[0]
        rows.append({"system": name, "subset": "clean (no overlap)", "n": len(idx),
                     **scores([r["hyp"][i] for i in idx], [in22_ref[i] for i in idx])})
score_df = pd.DataFrame(rows)
display(score_df)
score_df.to_csv(OUT_DIR / "scores_in22gen.csv", index=False)
print("\nFor orientation only: the MMLoSo companion paper reports 7.3 BLEU and 40.3 chrF++ on IN22-Gen (English->Santali) for IndicTrans2 fine-tuned on MMLoSo plus extra public data. "
      "Different training data and scoring details, so this is a rough sanity range, not a target.")

,system,subset,n,BLEU,chrF,chrF++,TER,word_len_ratio,empty
0,"pretrained, no adaptation",all,1024,4.82,34.72,31.36,104.33,1.210,0
1,"pretrained, no adaptation",clean (no overlap),1023,4.82,34.70,31.34,104.34,1.210,0
2,baseline seed 42,all,1024,5.88,37.53,33.75,96.53,1.144,0
3,baseline seed 42,clean (no overlap),1023,5.88,37.52,33.73,96.54,1.144,0
4,baseline seed 43,all,1024,5.92,37.51,33.77,96.82,1.143,0
5,baseline seed 43,clean (no overlap),1023,5.92,37.49,33.76,96.83,1.143,0
6,"mean init 445, seed 42",all,1024,6.06,38.18,34.43,94.98,1.139,1
7,"mean init 445, seed 42",clean (no overlap),1023,6.06,38.16,34.42,94.98,1.139,1
8,"mean init 445, seed 43",all,1024,6.32,38.33,34.59,93.75,1.128,1
9,"mean init 445, seed 43",clean (no overlap),1023,6.32,38.32,34.58,93.75,1.128,1



For orientation only: the MMLoSo companion paper reports 7.3 BLEU and 40.3 chrF++ on IN22-Gen (English->Santali) for IndicTrans2 fine-tuned on MMLoSo plus extra public data. Different training data and scoring details, so this is a rough sanity range, not a target.


## Section 11. Paired bootstrap
Each extended run against each baseline seed, on BLEU and chrF++, with 95% intervals over benchmark sentences (1,000 resamples). The fast path computes per-sentence statistics once and is verified against the normal scores; if the check fails it falls back to a slow method with fewer resamples. The intervals cover benchmark-sentence sampling only, not training randomness.

In [20]:
# Section 11.1: bootstrap
FAST, st = True, {}
try:
    from sacrebleu.metrics import BLEU, CHRF
    bleu_m, chrf_m = BLEU(), CHRF(word_order=2)
    for k, r in results.items():
        sb_ = bleu_m._extract_corpus_statistics(r["hyp"], [in22_ref]); sc_ = chrf_m._extract_corpus_statistics(r["hyp"], [in22_ref])
        assert abs(bleu_m._aggregate_and_compute(sb_).score - sacrebleu.corpus_bleu(r["hyp"], [in22_ref]).score) < 1e-6
        assert abs(chrf_m._aggregate_and_compute(sc_).score - sacrebleu.corpus_chrf(r["hyp"], [in22_ref], word_order=2).score) < 1e-6
        st[k] = (sb_, sc_)
    print("Fast bootstrap path verified.")
except Exception as e:
    FAST = False
    print("Fast path unavailable (" + repr(e)[:100] + "): slow method with 200 resamples.")
B = 1000 if FAST else 200

def boot(a, b, seed=0):
    rng = np.random.default_rng(seed); n = len(in22_ref); db, dc = [], []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        if FAST:
            db.append(bleu_m._aggregate_and_compute([st[b][0][i] for i in idx]).score - bleu_m._aggregate_and_compute([st[a][0][i] for i in idx]).score)
            dc.append(chrf_m._aggregate_and_compute([st[b][1][i] for i in idx]).score - chrf_m._aggregate_and_compute([st[a][1][i] for i in idx]).score)
        else:
            r_ = [in22_ref[i] for i in idx]; x = [results[a]["hyp"][i] for i in idx]; y = [results[b]["hyp"][i] for i in idx]
            db.append(sacrebleu.corpus_bleu(y, [r_]).score - sacrebleu.corpus_bleu(x, [r_]).score)
            dc.append(sacrebleu.corpus_chrf(y, [r_], word_order=2).score - sacrebleu.corpus_chrf(x, [r_], word_order=2).score)
    return db, dc

def fmt(d):
    lo, hi = np.percentile(d, [2.5, 97.5]); return f"{np.mean(d):+.2f} [{lo:+.2f}, {hi:+.2f}]"

baselines = [b for b in ("baseline seed 42", "baseline seed 43") if b in results]
ext_runs = [n for n in results if results[n]["n_new"] > 0]
rows = []
for run in ext_runs:
    for base in baselines:
        db, dc = boot(base, run)
        rows.append({"run": run, "vs": base, "dBLEU [95% CI]": fmt(db), "dchrF++ [95% CI]": fmt(dc)})
if len(baselines) == 2:
    db, dc = boot("baseline seed 42", "baseline seed 43")
    rows.append({"run": "baseline seed 43", "vs": "baseline seed 42 (seed spread)", "dBLEU [95% CI]": fmt(db), "dchrF++ [95% CI]": fmt(dc)})
if "pretrained, no adaptation" in results:
    for base in baselines:
        db, dc = boot("pretrained, no adaptation", base)
        rows.append({"run": base, "vs": "pretrained, no adaptation", "dBLEU [95% CI]": fmt(db), "dchrF++ [95% CI]": fmt(dc)})
boot_df = pd.DataFrame(rows)
display(boot_df)
boot_df.to_csv(OUT_DIR / "bootstrap_in22gen.csv", index=False)

Fast bootstrap path verified.


,run,vs,dBLEU [95% CI],dchrF++ [95% CI]
0,"mean init 445, seed 42",baseline seed 42,"+0.18 [-0.19, +0.54]","+0.69 [+0.25, +1.10]"
1,"mean init 445, seed 42",baseline seed 43,"+0.13 [-0.26, +0.51]","+0.66 [+0.23, +1.10]"
2,"mean init 445, seed 43",baseline seed 42,"+0.44 [+0.15, +0.73]","+0.85 [+0.45, +1.28]"
3,"mean init 445, seed 43",baseline seed 43,"+0.40 [+0.09, +0.70]","+0.82 [+0.43, +1.27]"
4,"random init 445, seed 42",baseline seed 42,"+0.10 [-0.25, +0.42]","+0.59 [+0.17, +1.00]"
5,"random init 445, seed 42",baseline seed 43,"+0.06 [-0.32, +0.46]","+0.56 [+0.08, +1.04]"
6,baseline seed 43,baseline seed 42 (seed spread),"+0.04 [-0.20, +0.29]","+0.03 [-0.37, +0.43]"
7,baseline seed 42,"pretrained, no adaptation","+1.06 [+0.62, +1.49]","+2.38 [+1.67, +3.03]"
8,baseline seed 43,"pretrained, no adaptation","+1.11 [+0.66, +1.54]","+2.41 [+1.71, +3.06]"


## Section 12. Does the difference sit where the added words are?
For the 445-token systems: the difference against each baseline on benchmark sentences whose reference contains an added word, and on those whose output uses an added token. In the MMLoSo test the gain concentrated in those sentences; this shows whether that pattern holds out of domain.

In [21]:
# Section 12.1: subset analysis
e445 = EXT[445]
ref_has = np.array([e445.pattern.search(r) is not None for r in in22_ref])
print(f"IN22-Gen references containing an added word (445 list): {int(ref_has.sum())} / {len(in22_ref)}")
rows = []
for run in [n for n in results if results[n]["n_new"] == 445]:
    out_has = np.array([any(t >= BASE for t in ids) for ids in results[run]["ids"]])
    for base in baselines:
        for label, mask in [("all", np.ones(len(in22_ref), dtype=bool)), ("reference has an added word", ref_has),
                            ("reference has NO added word", ~ref_has), ("output uses an added token", out_has),
                            ("output uses NO added token", ~out_has)]:
            idx = np.where(mask)[0]
            if len(idx) == 0:
                continue
            r_ = [in22_ref[i] for i in idx]; b_ = [results[base]["hyp"][i] for i in idx]; e_ = [results[run]["hyp"][i] for i in idx]
            rows.append({"run": run, "vs": base, "subset": label, "n": len(idx),
                         "dBLEU": round(sacrebleu.corpus_bleu(e_, [r_]).score - sacrebleu.corpus_bleu(b_, [r_]).score, 2),
                         "dchrF++": round(sacrebleu.corpus_chrf(e_, [r_], word_order=2).score - sacrebleu.corpus_chrf(b_, [r_], word_order=2).score, 2)})
subset_df = pd.DataFrame(rows)
display(subset_df)
subset_df.to_csv(OUT_DIR / "subset_in22gen.csv", index=False)

usage = []
for run, r in results.items():
    if r["n_new"] > 0:
        used = Counter(t for ids in r["ids"] for t in ids if t >= BASE)
        usage.append({"run": run, "sentences using an added token": int(sum(any(t >= BASE for t in ids) for ids in r["ids"])),
                      "added-token occurrences": int(sum(used.values())), "distinct added tokens used": len(used), "available": r["n_new"]})
display(pd.DataFrame(usage))

IN22-Gen references containing an added word (445 list): 532 / 1024


,run,vs,subset,n,dBLEU,dchrF++
0,"mean init 445, seed 42",baseline seed 42,all,1024,0.18,0.69
1,"mean init 445, seed 42",baseline seed 42,reference has an added word,532,0.07,0.76
2,"mean init 445, seed 42",baseline seed 42,reference has NO added word,492,0.33,0.58
3,"mean init 445, seed 42",baseline seed 42,output uses an added token,377,0.26,1.31
4,"mean init 445, seed 42",baseline seed 42,output uses NO added token,647,0.12,0.30
5,"mean init 445, seed 42",baseline seed 43,all,1024,0.14,0.66
6,"mean init 445, seed 42",baseline seed 43,reference has an added word,532,0.11,0.60
7,"mean init 445, seed 42",baseline seed 43,reference has NO added word,492,0.18,0.74
8,"mean init 445, seed 42",baseline seed 43,output uses an added token,377,0.55,1.12
9,"mean init 445, seed 42",baseline seed 43,output uses NO added token,647,-0.11,0.38


,run,sentences using an added token,added-token occurrences,distinct added tokens used,available
0,"mean init 445, seed 42",377,523,194,445
1,"mean init 445, seed 43",368,509,186,445
2,"random init 445, seed 42",312,407,177,445


## Section 13. How to read the results
* **Extended runs above both baseline seeds on chrF++ with intervals above zero:** the gain generalises to an independent, multi-domain benchmark, which strengthens the claim.
* **Differences near zero:** the benefit is domain-bound (the added words come from the training domain). Check the coverage table in Section 6: low coverage predicts a small effect, and this is itself a reportable finding.
* **All systems low and noisy:** IN22-Gen has about 1,000 sentences and published English→Santali scores are low, so differences below about 0.5 BLEU cannot be detected; rely on chrF++.
* **Pretrained system far above or below the fine-tuned ones:** compare with the scores of the original model carefully, since this checkpoint was not trained for English as a source language.

Report these results separately from the MMLoSo split results; do not average them together.

In [22]:
# Section 13.1: save everything
shutil.make_archive("/kaggle/working/in22gen_eval_results", "zip", OUT_DIR.parent, OUT_DIR.name)
print("Results bundle: /kaggle/working/in22gen_eval_results.zip (%.1f MB)" % (os.path.getsize("/kaggle/working/in22gen_eval_results.zip") / 1e6))
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(OUT_DIR), f"({p.stat().st_size/1024:.1f} KB)")

Results bundle: /kaggle/working/in22gen_eval_results.zip (2.3 MB)
 - bootstrap_in22gen.csv (0.8 KB)
 - coverage.csv (0.3 KB)
 - in22gen_en_sat.csv (593.6 KB)
 - predictions/baseline_seed_42.csv (1037.7 KB)
 - predictions/baseline_seed_42_generated_ids.json (498.4 KB)
 - predictions/baseline_seed_43.csv (1038.8 KB)
 - predictions/baseline_seed_43_generated_ids.json (497.7 KB)
 - predictions/mean_init_445_seed_42.csv (1036.6 KB)
 - predictions/mean_init_445_seed_42_generated_ids.json (482.2 KB)
 - predictions/mean_init_445_seed_43.csv (1031.5 KB)
 - predictions/mean_init_445_seed_43_generated_ids.json (467.3 KB)
 - predictions/pretrained_no_adaptation.csv (1053.4 KB)
 - predictions/pretrained_no_adaptation_generated_ids.json (536.1 KB)
 - predictions/random_init_445_seed_42.csv (1039.7 KB)
 - predictions/random_init_445_seed_42_generated_ids.json (485.6 KB)
 - scores_in22gen.csv (0.9 KB)
 - subset_in22gen.csv (2.4 KB)
